# Derivation of Hamiltonian and Necessary Conditions

In this notebook, we consider the following periodic optimal control problem:

$$
    \begin{align*}
        \max_{u} \quad & R = \frac{1}{T_{\text{lap}}(u)} \int_{0}^{T_{\text{lap}}(u)} \sum_{j=1}^N \gamma_j q_j(t) \, dt           \\
        \text{s.t.} \quad & \dot{s} = u(t)\\
        & s(0)=0,\; s(T_{\text{lap}}(u)) = L\\
        & \dot{q}_j(t) = S_j(s)(1 - q_j(t))^2 - \alpha q_j(t)^2, \nonumber\\ &\quad \forall j \in \{1, \dots, N\}\\
        & q_j(0) = q_j(T_{\text{lap}}(u)), \quad \forall j \in \{1, \dots, N\} \\
        & \frac{1}{T_{\text{lap}}(u)} \int_{0}^{T_{\text{lap}}(u)} P_{\mathrm{out}}(u(t)) \, dt = P_{\mathrm{in,avg}}.
    \end{align*}
$$

We begin by transcribing the objective and dynamics into Julia. We will be using the `Symbolics.jl` package for this analysis.

In [1]:
using Symbolics

We define all the variables for the problem:

In [11]:
# Define dimension N (keep it small for ease of derivation)
N_nodes = 2
@variables t

# Define states, costates, and control
@variables s(t) b(t) u(t)
@variables lambda_s(t) lambda_b(t)

# Define array states (q) and costates (lambda_q)
@variables q[1:N_nodes] lambda_q[1:N_nodes]

# Define parameters
@variables P_in_avg alpha t_f
@variables gamma[1:N_nodes]

# Define functions for P_out and S
@variables P_out(..) S(..)[1:N_nodes]


2-element Vector{Symbolics.CallAndWrap}:
 P_out⋆
 S⋆

We define the objective:

In [23]:
# Running objective
g_x = (1 / t_f) * sum(gamma[i] * q[i] for i in 1:N_nodes)

(gamma[1]*q[1] + gamma[2]*q[2]) / t_f

We define all the dynamics:

In [22]:
s_dot = u
b_dot = P_in_avg - P_out(u)

q_dot = [S(s)[i] * (1 - q[i])^2 - alpha * q[i]^2 for i in 1:N_nodes]

2-element Vector{Num}:
 (S(s(t)))[1]*((1 - q[1])^2) - alpha*(q[1]^2)
 (S(s(t)))[2]*((1 - q[2])^2) - alpha*(q[2]^2)

## Constructing the Hamiltonian

In [24]:
# Inner product of costates and dynamics 
inner_product = lambda_s * s_dot + lambda_b * b_dot + sum(lambda_q[i] * q_dot[i] for i in 1:N_nodes)

H = g_x + inner_product

println("=== Hamiltonian ===")
println("H = ", H)

=== Hamiltonian ===
H = (gamma[1]*q[1] + gamma[2]*q[2]) / t_f + (P_in_avg - P_out(u(t)))*lambda_b(t) + lambda_s(t)*u(t) + ((S(s(t)))[1]*((1 - q[1])^2) - alpha*(q[1]^2))*lambda_q[1] + ((S(s(t)))[2]*((1 - q[2])^2) - alpha*(q[2]^2))*lambda_q[2]


## Deriving First-Order Necessary Conditions

In [27]:
lambda_s_dot = -Symbolics.derivative(H, s)
lambda_b_dot = -Symbolics.derivative(H, b)
lambda_q_dot = [-Symbolics.derivative(H, q[i]) for i in 1:N_nodes]

# Control Optimality: dH/du = 0
dH_du = Symbolics.derivative(H, u)

println("=== Costate Dynamics ===")
println("d(lambda_s)/dt = ", lambda_s_dot)
println("d(lambda_b)/dt = ", lambda_b_dot)   # Outputs 0 -> lambda_b is constant
for i in 1:N_nodes
    println("d(lambda_q_$i)/dt = ", lambda_q_dot[i])
end
println()

println("=== Control Optimality Condition (dH/du = 0) ===")
println("dH/du = ", dH_du)
println()

=== Costate Dynamics ===
d(lambda_s)/dt = -Differential(s(t), 1)((S(s(t)))[1])*lambda_q[1]*((1 - q[1])^2) - Differential(s(t), 1)((S(s(t)))[2])*lambda_q[2]*((1 - q[2])^2)
d(lambda_b)/dt = 0
d(lambda_q_1)/dt = -(gamma[1] / t_f) - (-2(S(s(t)))[1]*(1 - q[1]) - 2alpha*q[1])*lambda_q[1]
d(lambda_q_2)/dt = -(gamma[2] / t_f) - (-2(S(s(t)))[2]*(1 - q[2]) - 2alpha*q[2])*lambda_q[2]

=== Control Optimality Condition (dH/du = 0) ===
dH/du = lambda_s(t) - Differential(u(t), 1)(P_out(u(t)))*lambda_b(t)

